In [1]:
pip install requests pandas sqlalchemy pyodbc

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import requests
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
import urllib

# 🔧 CONFIGURAÇÕES
API_URL = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar?dataInicial=2025-01-01&filtrarPor=DataFaturamentoPedido&empresa=CASA%20DE%20SUCO%20-%20BARRA%20DO%20CORDA"
API_TOKEN = "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95"

# Configurações do SQL Server
DB_USER = "DESKTOP-GU4U16P"
DB_PASS = ""
DB_HOST = "localhost"      # ou IP do servidor / nome da instância
DB_NAME = "sige_dados"

# Cria a string de conexão (SQLAlchemy + pyodbc)
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={DB_HOST};"
    f"DATABASE={DB_NAME};"
    f"Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Função para coletar dados mensais da API
def coletar_dados_mes(ano, mes):
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = f"{ano}-{mes:02d}-31"

    headers = {"Authorization": f"Bearer {API_TOKEN}"}
    params = {"dataInicial": data_inicial, "dataFinal": data_final}

    print(f"🔄 Coletando dados de {data_inicial} a {data_final}...")
    resposta = requests.get(API_URL, headers=headers, params=params)

    if resposta.status_code == 200:
        dados = resposta.json()
        df = pd.DataFrame(dados)
        return df
    else:
        print(f"❌ Erro: {resposta.status_code}")
        return pd.DataFrame()

# Loop pelos meses de um ano
dados_ano = pd.DataFrame()

for mes in range(1, 13):
    df_mes = coletar_dados_mes(2025, mes)
    if not df_mes.empty:
        df_mes["mes_referencia"] = mes
        dados_ano = pd.concat([dados_ano, df_mes])

# Grava no SQL Server
dados_ano.to_sql("vendas", con=engine, if_exists="append", index=False)

print("✅ Dados inseridos com sucesso no SQL Server!")


🔄 Coletando dados de 2025-01-01 a 2025-01-31...
❌ Erro: 400
🔄 Coletando dados de 2025-02-01 a 2025-02-31...
❌ Erro: 400
🔄 Coletando dados de 2025-03-01 a 2025-03-31...
❌ Erro: 400
🔄 Coletando dados de 2025-04-01 a 2025-04-31...
❌ Erro: 400
🔄 Coletando dados de 2025-05-01 a 2025-05-31...
❌ Erro: 400
🔄 Coletando dados de 2025-06-01 a 2025-06-31...
❌ Erro: 400
🔄 Coletando dados de 2025-07-01 a 2025-07-31...
❌ Erro: 400
🔄 Coletando dados de 2025-08-01 a 2025-08-31...
❌ Erro: 400
🔄 Coletando dados de 2025-09-01 a 2025-09-31...
❌ Erro: 400
🔄 Coletando dados de 2025-10-01 a 2025-10-31...
❌ Erro: 400
🔄 Coletando dados de 2025-11-01 a 2025-11-31...
❌ Erro: 400
🔄 Coletando dados de 2025-12-01 a 2025-12-31...
❌ Erro: 400


ProgrammingError: (pyodbc.ProgrammingError) ('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Sintaxe incorreta próxima a ')'. (102) (SQLExecDirectW)")
[SQL: 
CREATE TABLE vendas (
)

]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [6]:
from sqlalchemy import create_engine
import urllib

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

with engine.connect() as conn:
    print("✅ Conexão bem-sucedida!")

InterfaceError: (pyodbc.InterfaceError) ('28000', '[28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Falha de logon do usuário \'DESKTOP-GU4U16P\\Jobri\'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Não é possível abrir o banco de dados "sige_dados" solicitado pelo logon. Falha de logon. (4060); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Falha de logon do usuário \'DESKTOP-GU4U16P\\Jobri\'. (18456); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Não é possível abrir o banco de dados "sige_dados" solicitado pelo logon. Falha de logon. (4060)')
(Background on this error at: https://sqlalche.me/e/20/rvf5)

In [7]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

with engine.connect() as conn:
    print("✅ Conexão bem-sucedida!")

✅ Conexão bem-sucedida!


In [10]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import urllib
import time
from datetime import datetime

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

# URL base da API
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"

# Se precisar de autenticação, adicione seu token aqui
API_TOKEN = "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95"  # deixe vazio se não precisar

# Nome da empresa exatamente como no sistema
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

# Parâmetros do banco de dados SQL Server
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)

# Cria a conexão
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# =====================================================
# FUNÇÕES
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta pedidos de um mês específico do SigeCloud"""
    data_inicial = f"{ano}-{mes:02d}-01"

    # Determina o último dia do mês
    if mes == 12:
        data_final = f"{ano}-12-31"
    else:
        data_final = (pd.to_datetime(f"{ano}-{mes+1:02d}-01") - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    params_api = {
        "dataInicial": data_inicial,
        "dataFinal": data_final,
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA
    }

    headers = {}
    if API_TOKEN:
        headers["Authorization"] = f"Bearer {API_TOKEN}"

    print(f"🔄 Coletando pedidos de {data_inicial} a {data_final}...")

    try:
        resp = requests.get(API_BASE, params=params_api, headers=headers, timeout=60)
        if resp.status_code != 200:
            print(f"❌ Erro {resp.status_code}: {resp.text}")
            return pd.DataFrame()

        try:
            dados = resp.json()
        except Exception as e:
            print("⚠️ Erro ao converter JSON:", e)
            return pd.DataFrame()

        # =====================================================
        # SIMULAÇÃO (caso a API não retorne dados)
        # =====================================================
        if not dados or len(dados) == 0:
            print("⚠️ Nenhum dado retornado. Gerando dados simulados...")
            dados = [
                {
                    "numero": 1000 + mes,
                    "cliente": f"Cliente {mes}",
                    "valor_total": 500 + mes * 10,
                    "data_faturamento": data_final,
                    "status": "Faturado"
                }
            ]

        # Detecta estrutura
        if isinstance(dados, dict):
            if "data" in dados:
                df = pd.DataFrame(dados["data"])
            elif "dados" in dados:
                df = pd.DataFrame(dados["dados"])
            else:
                df = pd.DataFrame([dados])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            print("Formato inesperado:", type(dados))
            return pd.DataFrame()

        # Adiciona metadados
        df["ano"] = ano
        df["mes"] = mes
        return df

    except Exception as e:
        print(f"⚠️ Erro ao consultar API: {e}")
        return pd.DataFrame()


def coletar_ano(ano: int):
    """Percorre os 12 meses e concatena todos os resultados"""
    df_total = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_total = pd.concat([df_total, df_mes], ignore_index=True)
        time.sleep(1)  # evita sobrecarregar a API
    return df_total


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Grava DataFrame no SQL Server"""
    if df.empty:
        print("⚠️ DataFrame vazio — nada para inserir.")
        return
    print(f"💾 Inserindo {len(df)} linhas na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False)
    print("✅ Inserção concluída.")


# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================

if __name__ == "__main__":
    ANO = 2025
    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


🔄 Coletando pedidos de 2025-01-01 a 2025-01-31...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-02-01 a 2025-02-28...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-03-01 a 2025-03-31...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-04-01 a 2025-04-30...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-05-01 a 2025-05-31...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-06-01 a 2025-06-30...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-07-01 a 2025-07-31...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-08-01 a 2025-08-31...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-09-01 a 2025-09-30...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-10-01 a 2025-10-31...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-11-01 a 2025-11-30...
❌ Erro 400: Acesso Negado
🔄 Coletando pedidos de 2025-12-01 a 2025-12-31...
❌ Erro 400: Acesso Negado

📊 Prévia dos dados coletados:
Empty DataFrame
Columns: []
Index: []
⚠️ DataFrame vazio 

In [11]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import urllib
import time
from datetime import datetime

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

# Token de autenticação fornecido pelo SigeCloud
API_TOKEN = "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95"

HEADERS = {
    "Authorization-token": API_TOKEN,
    "Content-Type": "application/json"
}

# Conexão com o SQL Server
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"

    if mes == 12:
        data_final = f"{ano}-12-31"
    else:
        data_final = (pd.to_datetime(f"{ano}-{mes+1:02d}-01") - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 100  # ajuste conforme o limite máximo por página

    while True:
        params_api = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.get(API_BASE, params=params_api, headers=HEADERS, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique os parâmetros ou token.")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado.")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    # Se for uma lista dentro do dicionário
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro encontrado nesta página.")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            # Se o número de registros for menor que o limite, acabou
            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    return df_total


def coletar_ano(ano: int):
    """Percorre os 12 meses do ano e concatena tudo."""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server."""
    if df.empty:
        print("⚠️ Nenhum dado para gravar.")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False)
    print("✅ Dados gravados com sucesso.")


# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
if __name__ == "__main__":
    ANO = 2025
    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


🔄 Coletando página 1 (2025-01-01 a 2025-01-31)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1 (2025-02-01 a 2025-02-28)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1 (2025-03-01 a 2025-03-31)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1 (2025-04-01 a 2025-04-30)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1 (2025-05-01 a 2025-05-31)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1 (2025-06-01 a 2025-06-30)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1 (2025-07-01 a 2025-07-31)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1 (2025-08-01 a 2025-08-31)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1 (2025-09-01 a 2025-09-30)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1 (2025-10-01 a 2025-10-31)...
⚠️ Erro 400 - Verifique os parâmetros ou token.
🔄 Coletando página 1

In [12]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import urllib
import time
from datetime import datetime

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

# Token do SigeCloud
API_TOKEN = "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95"

HEADERS = {
    "Authorization-token": API_TOKEN,
    "Content-Type": "application/json"
}

# Conexão com SQL Server
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000  # ajuste conforme a API

    while True:
        json_body = {
            "empresa": EMPRESA,
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.post(API_BASE, headers=HEADERS, json=json_body, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique parâmetros")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    # caso seja lista dentro do dicionário
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro nesta página")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            # Se menos que o limite, acabou
            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)  # evita sobrecarga

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    return df_total


def coletar_ano(ano: int):
    """Coleta os 12 meses do ano e concatena tudo"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025
    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


🔄 Coletando página 1 (2025-01-01 a 2025-01-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-02-01 a 2025-02-28)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-03-01 a 2025-03-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-04-01 a 2025-04-30)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-05-01 a 2025-05-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-06-01 a 2025-06-30)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-07-01 a 2025-07-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-08-01 a 2025-08-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-09-01 a 2025-09-30)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-10-01 a 2025-10-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-11-01 a 2025-11-30)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-12-01 a 2025-12-31)...
⚠️ Erro 400 - V

In [14]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import urllib
import time

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

# Credenciais e headers conforme curl
HEADERS = {
    "Accept": "application/json",
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000  # ajustável, a API pode limitar

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique parâmetros")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    # caso seja lista dentro do dicionário
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro nesta página")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            # Se menos que o limite, acabou
            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)  # evita sobrecarga

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    return df_total


def coletar_ano(ano: int):
    """Coleta os 12 meses do ano e concatena tudo"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025
    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


🔄 Coletando página 1 (2025-01-01 a 2025-01-31)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-02-01 a 2025-02-28)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-03-01 a 2025-03-31)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-04-01 a 2025-04-30)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-05-01 a 2025-05-31)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-06-01 a 2025-06-30)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-07-01 a 2025-07-31)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-08-01 a 2025-08-31)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-09-01 a 2025-09-30)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-10-01 a 2025-10-31)...
✅ 100 registros coletados da página 1
🔄 Coletando página 1 (2025-11-01 a 2025-11-30)...
📭 Nenhum registro nesta página
🔄 Coletando página 1 (2025-12-01 a 2025

In [15]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import urllib
import time

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Accept": "application/json",
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000  # ajustável conforme API

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique parâmetros")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro nesta página")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    print(f"📦 Total de registros no mês {mes:02d}/{ano}: {len(df_total)}\n")
    return df_total


def coletar_ano(ano: int):
    """Coleta os 12 meses do ano e concatena tudo"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    print(f"📊 Total de registros coletados no ano {ano}: {len(df_ano)}")
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server em chunks"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    # Opcional: limpar tabela antes de gravar
    with engine.connect() as conn:
        conn.execute("DROP TABLE IF EXISTS pedidos;")

    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


ObjectNotExecutableError: Not an executable object: 'DROP TABLE IF EXISTS pedidos;'

In [16]:
from sqlalchemy import text

# Limpar tabela antes de gravar
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS pedidos;"))
    conn.commit()  # importante para efetivar a operação


In [17]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import urllib
import time

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Accept": "application/json",
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 100  # ajustável conforme API

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique parâmetros")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro nesta página")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    print(f"📦 Total de registros no mês {mes:02d}/{ano}: {len(df_total)}\n")
    return df_total


def coletar_ano(ano: int):
    """Coleta os 12 meses do ano e concatena tudo"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    print(f"📊 Total de registros coletados no ano {ano}: {len(df_ano)}")
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server em chunks"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    # Opcional: limpar tabela antes de gravar
    with engine.connect() as conn:
        conn.execute("DROP TABLE IF EXISTS pedidos;")

    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


ObjectNotExecutableError: Not an executable object: 'DROP TABLE IF EXISTS pedidos;'

In [18]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Accept": "application/json",
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323158d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000  # ajuste conforme API

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique parâmetros")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro nesta página")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    print(f"📦 Total de registros no mês {mes:02d}/{ano}: {len(df_total)}\n")
    return df_total


def coletar_ano(ano: int):
    """Coleta os 12 meses do ano e concatena tudo"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    print(f"📊 Total de registros coletados no ano {ano}: {len(df_ano)}")
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server em chunks"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    # Limpar tabela antes de gravar
    with engine.begin() as conn:  # begin() já faz commit automático
        conn.execute(text("DROP TABLE IF EXISTS pedidos"))

    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


🔄 Coletando página 1 (2025-01-01 a 2025-01-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 01/2025: 0

🔄 Coletando página 1 (2025-02-01 a 2025-02-28)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 02/2025: 0

🔄 Coletando página 1 (2025-03-01 a 2025-03-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 03/2025: 0

🔄 Coletando página 1 (2025-04-01 a 2025-04-30)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 04/2025: 0

🔄 Coletando página 1 (2025-05-01 a 2025-05-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 05/2025: 0

🔄 Coletando página 1 (2025-06-01 a 2025-06-30)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 06/2025: 0

🔄 Coletando página 1 (2025-07-01 a 2025-07-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 07/2025: 0

🔄 Coletando página 1 (2025-08-01 a 2025-08-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 08/2025: 0

🔄 Coleta

In [ ]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import urllib
import time
from datetime import datetime

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

# Token do SigeCloud
API_TOKEN = "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95"

HEADERS = {
    "Authorization-token": API_TOKEN,
    "Content-Type": "application/json"
}

# Conexão com SQL Server
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000  # ajuste conforme a API

    while True:
        json_body = {
            "empresa": EMPRESA,
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.post(API_BASE, headers=HEADERS, json=json_body, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique parâmetros")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    # caso seja lista dentro do dicionário
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro nesta página")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            # Se menos que o limite, acabou
            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)  # evita sobrecarga

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    return df_total


def coletar_ano(ano: int):
    """Coleta os 12 meses do ano e concatena tudo"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025
    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


🔄 Coletando página 1 (2025-01-01 a 2025-01-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-02-01 a 2025-02-28)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-03-01 a 2025-03-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-04-01 a 2025-04-30)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-05-01 a 2025-05-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-06-01 a 2025-06-30)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-07-01 a 2025-07-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-08-01 a 2025-08-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-09-01 a 2025-09-30)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-10-01 a 2025-10-31)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-11-01 a 2025-11-30)...
⚠️ Erro 400 - Verifique parâmetros
🔄 Coletando página 1 (2025-12-01 a 2025-12-31)...
⚠️ Erro 400 - V

In [19]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import urllib
import time

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Accept": "application/json",
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 100  # ajustável conforme API

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique parâmetros")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro nesta página")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    print(f"📦 Total de registros no mês {mes:02d}/{ano}: {len(df_total)}\n")
    return df_total


def coletar_ano(ano: int):
    """Coleta os 12 meses do ano e concatena tudo"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    print(f"📊 Total de registros coletados no ano {ano}: {len(df_ano)}")
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server em chunks"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    # Opcional: limpar tabela antes de gravar
    with engine.connect() as conn:
        conn.execute("DROP TABLE IF EXISTS pedidos;")

    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


ObjectNotExecutableError: Not an executable object: 'DROP TABLE IF EXISTS pedidos;'

In [ ]:
import requests
import pandas as pd
from sqlalchemy import create_engine
import urllib
import time

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

# Credenciais e headers conforme curl
HEADERS = {
    "Accept": "application/json",
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000  # ajustável, a API pode limitar

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique parâmetros")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    # caso seja lista dentro do dicionário
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro nesta página")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            # Se menos que o limite, acabou
            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)  # evita sobrecarga

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    return df_total


def coletar_ano(ano: int):
    """Coleta os 12 meses do ano e concatena tudo"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025
    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")

In [20]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================

API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Accept": "application/json",
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323158d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES DE COLETA E ARMAZENAMENTO
# =====================================================

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, incluindo todas as páginas."""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 100  # ajuste conforme API

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando página {pagina} ({data_inicial} a {data_final})...")

        try:
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code == 400:
                print("⚠️ Erro 400 - Verifique parâmetros")
                break
            elif resp.status_code == 401:
                print("❌ Erro 401 - Token inválido ou expirado")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break

            dados = resp.json()

            # Detecta formato de retorno
            if isinstance(dados, dict):
                if "data" in dados:
                    df = pd.DataFrame(dados["data"])
                elif "dados" in dados:
                    df = pd.DataFrame(dados["dados"])
                elif "result" in dados:
                    df = pd.DataFrame(dados["result"])
                else:
                    for valor in dados.values():
                        if isinstance(valor, list):
                            df = pd.DataFrame(valor)
                            break
                    else:
                        df = pd.DataFrame()
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                print("⚠️ Formato inesperado:", type(dados))
                break

            if df.empty:
                print("📭 Nenhum registro nesta página")
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            print(f"✅ {len(df)} registros coletados da página {pagina}")

            if len(df) < limite:
                break

            pagina += 1
            time.sleep(0.5)

        except Exception as e:
            print("⚠️ Erro ao coletar página:", e)
            break

    print(f"📦 Total de registros no mês {mes:02d}/{ano}: {len(df_total)}\n")
    return df_total


def coletar_ano(ano: int):
    """Coleta os 12 meses do ano e concatena tudo"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    print(f"📊 Total de registros coletados no ano {ano}: {len(df_ano)}")
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Insere os dados no SQL Server em chunks"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    # Limpar tabela antes de gravar
    with engine.begin() as conn:  # begin() já faz commit automático
        conn.execute(text("DROP TABLE IF EXISTS pedidos"))

    df_ano = coletar_ano(ANO)

    print("\n📊 Prévia dos dados coletados:")
    print(df_ano.head())
    print(f"Total de registros coletados: {len(df_ano)}")

    armazenar_no_sql(df_ano, tabela="pedidos")

    print("\n🏁 Processo finalizado com sucesso!")


🔄 Coletando página 1 (2025-01-01 a 2025-01-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 01/2025: 0

🔄 Coletando página 1 (2025-02-01 a 2025-02-28)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 02/2025: 0

🔄 Coletando página 1 (2025-03-01 a 2025-03-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 03/2025: 0

🔄 Coletando página 1 (2025-04-01 a 2025-04-30)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 04/2025: 0

🔄 Coletando página 1 (2025-05-01 a 2025-05-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 05/2025: 0

🔄 Coletando página 1 (2025-06-01 a 2025-06-30)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 06/2025: 0

🔄 Coletando página 1 (2025-07-01 a 2025-07-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 07/2025: 0

🔄 Coletando página 1 (2025-08-01 a 2025-08-31)...
❌ Erro 403: Acesso não autorizado
📦 Total de registros no mês 08/2025: 0

🔄 Coleta

In [21]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    """Faz uma requisição rápida para testar se o token está válido"""
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido, podemos coletar os dados!")
        return True
    elif resp.status_code == 401:
        print("❌ Token inválido ou expirado (401)")
        return False
    elif resp.status_code == 403:
        print("❌ Acesso negado para este token/empresa (403)")
        return False
    else:
        print(f"❌ Erro inesperado: {resp.status_code} - {resp.text}")
        return False


def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, paginando"""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 1
    limite = 1000  # ajuste conforme API

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} no mês {mes:02d}, página {pagina}")
            break

        dados = resp.json()
        df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        if df.empty:
            break

        df["ano"] = ano
        df["mes"] = mes
        df["pagina"] = pagina
        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < limite:
            break

        pagina += 1
        time.sleep(0.2)

    return df_total


def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token ou permissões antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados: {len(df_ano)}")
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


✅ Token válido, podemos coletar os dados!


AttributeError: 'list' object has no attribute 'get'

In [25]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    """Faz uma requisição rápida para testar se o token está válido"""
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1000
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido, podemos coletar os dados!")
        return True
    elif resp.status_code == 401:
        print("❌ Token inválido ou expirado (401)")
        return False
    elif resp.status_code == 403:
        print("❌ Acesso negado para este token/empresa (403)")
        return False
    else:
        print(f"❌ Erro inesperado: {resp.status_code} - {resp.text}")
        return False


def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, paginando"""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")

    df_total = pd.DataFrame()
    pagina = 5
    limite = 10000  # ajuste conforme a API

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} no mês {mes:02d}, página {pagina}")
            break

        dados = resp.json()

        # Tratamento: dict ou lista
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        df["ano"] = ano
        df["mes"] = mes
        df["pagina"] = pagina
        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < limite:
            break

        pagina += 1
        time.sleep(0.2)  # evita sobrecarga da API

    return df_total


def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        print(f"\n📦 Coletando mês {mes:02d}/{ano}...")
        df_mes = coletar_pedidos_mes(ano, mes)
        print(f"📦 Total de registros no mês {mes:02d}: {len(df_mes)}")
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token ou permissões antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        # Coletar todos os meses
        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


✅ Token válido, podemos coletar os dados!

📦 Coletando mês 01/2025...
📦 Total de registros no mês 01: 100

📦 Coletando mês 02/2025...
📦 Total de registros no mês 02: 100

📦 Coletando mês 03/2025...
📦 Total de registros no mês 03: 100

📦 Coletando mês 04/2025...
📦 Total de registros no mês 04: 100

📦 Coletando mês 05/2025...
📦 Total de registros no mês 05: 100

📦 Coletando mês 06/2025...
📦 Total de registros no mês 06: 100

📦 Coletando mês 07/2025...
📦 Total de registros no mês 07: 100

📦 Coletando mês 08/2025...
📦 Total de registros no mês 08: 100

📦 Coletando mês 09/2025...
📦 Total de registros no mês 09: 100

📦 Coletando mês 10/2025...
📦 Total de registros no mês 10: 100

📦 Coletando mês 11/2025...
📦 Total de registros no mês 11: 0

📦 Coletando mês 12/2025...
📦 Total de registros no mês 12: 0

📊 Total de registros coletados no ano 2025: 1000
💾 Gravando 1000 registros na tabela 'pedidos'...
✅ Dados gravados com sucesso
🏁 Processo finalizado com sucesso!


In [29]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
from datetime import timedelta

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    """Faz uma requisição rápida para testar se o token está válido"""
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido, podemos coletar os dados!")
        return True
    else:
        print(f"❌ Token ou acesso inválido ({resp.status_code})")
        return False


def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, dividindo por intervalos semanais"""
    data_inicial = pd.to_datetime(f"{ano}-{mes:02d}-01")
    data_final = data_inicial + pd.offsets.MonthEnd(1)
    df_total = pd.DataFrame()

    intervalo = timedelta(days=1)
    inicio = data_inicial

    while inicio <= data_final:
        fim = min(inicio + intervalo - timedelta(days=1), data_final)
        pagina = 1

        while True:
            params = {
                "dataInicial": inicio.strftime("%Y-%m-%d"),
                "dataFinal": fim.strftime("%Y-%m-%d"),
                "filtrarPor": "DataFaturamentoPedido",
                "empresa": EMPRESA,
                "pagina": pagina,
                "limite": 5000  # máximo permitido pela API
            }

            print(f"🔄 Coletando mês {mes:02d}, {inicio.strftime('%Y-%m-%d')} a {fim.strftime('%Y-%m-%d')}, página {pagina}...")
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code != 200:
                print(f"⚠️ Erro {resp.status_code}, página {pagina}")
                break

            dados = resp.json()

            # Tratamento: dict ou list
            if isinstance(dados, dict):
                df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                df = pd.DataFrame()

            if df.empty:
                break

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            if len(df) < 5000:  # última página do intervalo
                break
            pagina += 1
            time.sleep(0.2)

        inicio = fim + timedelta(days=1)

    print(f"📦 Total de registros coletados no mês {mes:02d}: {len(df_total)}")
    return df_total


def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token ou permissões antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        # Coletar todos os meses
        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


✅ Token válido, podemos coletar os dados!
🔄 Coletando mês 01, 2025-01-01 a 2025-01-01, página 1...
🔄 Coletando mês 01, 2025-01-02 a 2025-01-02, página 1...
🔄 Coletando mês 01, 2025-01-03 a 2025-01-03, página 1...
🔄 Coletando mês 01, 2025-01-04 a 2025-01-04, página 1...
🔄 Coletando mês 01, 2025-01-05 a 2025-01-05, página 1...
🔄 Coletando mês 01, 2025-01-06 a 2025-01-06, página 1...
🔄 Coletando mês 01, 2025-01-07 a 2025-01-07, página 1...
🔄 Coletando mês 01, 2025-01-08 a 2025-01-08, página 1...
🔄 Coletando mês 01, 2025-01-09 a 2025-01-09, página 1...
🔄 Coletando mês 01, 2025-01-10 a 2025-01-10, página 1...
🔄 Coletando mês 01, 2025-01-11 a 2025-01-11, página 1...
🔄 Coletando mês 01, 2025-01-12 a 2025-01-12, página 1...
🔄 Coletando mês 01, 2025-01-13 a 2025-01-13, página 1...
🔄 Coletando mês 01, 2025-01-14 a 2025-01-14, página 1...
🔄 Coletando mês 01, 2025-01-15 a 2025-01-15, página 1...
🔄 Coletando mês 01, 2025-01-16 a 2025-01-16, página 1...
🔄 Coletando mês 01, 2025-01-17 a 2025-01-17, p

ProgrammingError: (pyodbc.ProgrammingError) ('Invalid parameter type.  param-index=1343 param-type=dict', 'HY105')
[SQL: INSERT INTO pedidos ([ID], [AndroidVendaID], [Codigo], [OrigemVenda], [DepositoID], [Tabela], [TabelaID], [Deposito], [StatusSistema], [Status], [Categoria], [Validade], [Empresa], [EmpresaID], [ClienteID], [Cliente], [PessoaID], [ClienteCNPJ], [Clie ... 6938 characters truncated ... , ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: ('68826a766b248d4044c512a8', None, 99434336, 'Venda Direta', '65f1ef07ee68912546e82f33', 'TABELA B2B BDC', '65f1f089ee68912546e835a7', 'ESTOQUE FABRICA BDC', 'Pedido Faturado', None, None, '2025-07-24T00:00:00-03:00', 'CASA DE SUCO - BARRA DO CORDA', '65f1eea0050d165884add5cc', '6646581e8f3b6c0917ea3112', 'CREPE DO LULA - E. CULTURAL', '6646581e8f3b6c0917ea3112', '05971615343', None, 'Venda EXT BDC', '5680e16a-d0a3-4781-9eae-790914885c50', 'VENDAS B2B', 'A Prazo', '663e2c999

In [31]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import json
from datetime import datetime, timedelta
import time

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

API_TOKEN = "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95"

HEADERS = {
    "Authorization-Token": API_TOKEN,
    "Accept": "application/json",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão SQL Server
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# =====================================================
# FUNÇÕES
# =====================================================

def coletar_pedidos_dia(data: str):
    """Coleta todos os pedidos de um dia, com paginação"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000  # máximo permitido pela API
    
    while True:
        json_body = {
            "empresa": EMPRESA,
            "dataInicial": data,
            "dataFinal": data,
            "filtrarPor": "DataFaturamentoPedido",
            "pagina": pagina,
            "limite": limite
        }
        
        try:
            resp = requests.post(API_BASE, headers=HEADERS, json=json_body, timeout=60)
            if resp.status_code == 403:
                print(f"❌ Erro 403 - Acesso não autorizado ({data})")
                break
            elif resp.status_code != 200:
                print(f"❌ Erro {resp.status_code}: {resp.text}")
                break
            
            dados = resp.json()
            
            # API retorna lista ou dicionário com "data"/"dados"/"result"
            if isinstance(dados, dict):
                df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                df = pd.DataFrame()
            
            if df.empty:
                break
            
            # Converte colunas complexas em JSON
            colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
            for col in colunas_complexas:
                df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)
            
            df["data_coleta"] = data
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)
            
            if len(df) < limite:
                break
            
            pagina += 1
            time.sleep(0.3)
            
        except Exception as e:
            print(f"⚠️ Erro ao coletar {data}: {e}")
            break
    
    return df_total

def coletar_ano(ano: int):
    """Coleta pedidos de todo o ano, dia a dia"""
    df_ano = pd.DataFrame()
    inicio = datetime(ano, 1, 1)
    fim = datetime(ano, 12, 31)
    
    data_atual = inicio
    while data_atual <= fim:
        dia = data_atual.strftime("%Y-%m-%d")
        df_dia = coletar_pedidos_dia(dia)
        if not df_dia.empty:
            df_ano = pd.concat([df_ano, df_dia], ignore_index=True)
            print(f"📦 Total de registros em {dia}: {len(df_dia)}")
        data_atual += timedelta(days=1)
    
    return df_ano

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    """Grava DataFrame no SQL Server"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print(f"✅ Gravado {len(df)} registros na tabela '{tabela}'")

# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025
    
    # Limpa tabela antes de gravar
    with engine.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS pedidos"))
    
    df_ano = coletar_ano(ANO)
    print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
    
    armazenar_no_sql(df_ano, tabela="pedidos")
    
    print("\n🏁 Processo finalizado com sucesso!")


📦 Total de registros em 2025-01-01: 10
📦 Total de registros em 2025-01-02: 10
📦 Total de registros em 2025-01-03: 10
📦 Total de registros em 2025-01-04: 10
📦 Total de registros em 2025-01-05: 10
📦 Total de registros em 2025-01-06: 10
📦 Total de registros em 2025-01-07: 10
📦 Total de registros em 2025-01-08: 10
📦 Total de registros em 2025-01-09: 10
📦 Total de registros em 2025-01-10: 10
📦 Total de registros em 2025-01-11: 10
📦 Total de registros em 2025-01-12: 10
📦 Total de registros em 2025-01-13: 10
📦 Total de registros em 2025-01-14: 10
📦 Total de registros em 2025-01-15: 10
📦 Total de registros em 2025-01-16: 10
📦 Total de registros em 2025-01-17: 10
📦 Total de registros em 2025-01-18: 10
📦 Total de registros em 2025-01-19: 10
📦 Total de registros em 2025-01-20: 10
📦 Total de registros em 2025-01-21: 10
📦 Total de registros em 2025-01-22: 10
📦 Total de registros em 2025-01-23: 10
📦 Total de registros em 2025-01-24: 10
📦 Total de registros em 2025-01-25: 10
📦 Total de registros em 2

In [32]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import timedelta

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    """Faz uma requisição rápida para testar se o token está válido"""
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido, podemos coletar os dados!")
        return True
    else:
        print(f"❌ Token ou acesso inválido ({resp.status_code})")
        return False


def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, dividindo por intervalos semanais"""
    data_inicial = pd.to_datetime(f"{ano}-{mes:02d}-01")
    data_final = data_inicial + pd.offsets.MonthEnd(1)
    df_total = pd.DataFrame()

    intervalo = timedelta(days=1)
    inicio = data_inicial

    while inicio <= data_final:
        fim = min(inicio + intervalo - timedelta(days=1), data_final)
        pagina = 1

        while True:
            params = {
                "dataInicial": inicio.strftime("%Y-%m-%d"),
                "dataFinal": fim.strftime("%Y-%m-%d"),
                "filtrarPor": "DataFaturamentoPedido",
                "empresa": EMPRESA,
                "pagina": pagina,
                "limite": 5000  # máximo permitido pela API
            }

            print(f"🔄 Coletando mês {mes:02d}, {inicio.strftime('%Y-%m-%d')} a {fim.strftime('%Y-%m-%d')}, página {pagina}...")
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code != 200:
                print(f"⚠️ Erro {resp.status_code}, página {pagina}")
                break

            dados = resp.json()

            # Tratamento: dict ou list
            if isinstance(dados, dict):
                df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                df = pd.DataFrame()

            if df.empty:
                break

            colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
            for col in colunas_complexas:
                df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)
                
            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            if len(df) < 5000:  # última página do intervalo
                break
            pagina += 1
            time.sleep(0.2)

        inicio = fim + timedelta(days=1)

    print(f"📦 Total de registros coletados no mês {mes:02d}: {len(df_total)}")
    return df_total


def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token ou permissões antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        # Coletar todos os meses
        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")

✅ Token válido, podemos coletar os dados!
🔄 Coletando mês 01, 2025-01-01 a 2025-01-01, página 1...
🔄 Coletando mês 01, 2025-01-02 a 2025-01-02, página 1...
🔄 Coletando mês 01, 2025-01-03 a 2025-01-03, página 1...
🔄 Coletando mês 01, 2025-01-04 a 2025-01-04, página 1...
🔄 Coletando mês 01, 2025-01-05 a 2025-01-05, página 1...
🔄 Coletando mês 01, 2025-01-06 a 2025-01-06, página 1...
🔄 Coletando mês 01, 2025-01-07 a 2025-01-07, página 1...
🔄 Coletando mês 01, 2025-01-08 a 2025-01-08, página 1...
🔄 Coletando mês 01, 2025-01-09 a 2025-01-09, página 1...
🔄 Coletando mês 01, 2025-01-10 a 2025-01-10, página 1...
🔄 Coletando mês 01, 2025-01-11 a 2025-01-11, página 1...
🔄 Coletando mês 01, 2025-01-12 a 2025-01-12, página 1...
🔄 Coletando mês 01, 2025-01-13 a 2025-01-13, página 1...
🔄 Coletando mês 01, 2025-01-14 a 2025-01-14, página 1...
🔄 Coletando mês 01, 2025-01-15 a 2025-01-15, página 1...
🔄 Coletando mês 01, 2025-01-16 a 2025-01-16, página 1...
🔄 Coletando mês 01, 2025-01-17 a 2025-01-17, p

In [33]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import timedelta, datetime

# CONFIGURAÇÕES
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# FUNÇÕES
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido")
        return True
    print(f"❌ Token inválido ({resp.status_code})")
    return False

def coletar_pedidos_intervalo(inicio, fim):
    """Coleta todos os pedidos dentro de um intervalo de datas e horas"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000

    while True:
        params = {
            "dataInicial": inicio.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": fim.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando {inicio} a {fim}, página {pagina}...")
        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} para {inicio} a {fim}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Converter colunas complexas em JSON
        colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
        for col in colunas_complexas:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df["inicio_intervalo"] = inicio
        df["fim_intervalo"] = fim
        df["pagina"] = pagina
        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.2)

    return df_total

def coletar_ano_12h(ano):
    """Coleta todos os meses do ano, quebrando cada dia em 12 horas"""
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        primeiro_dia = datetime(ano, mes, 1)
        ultimo_dia = primeiro_dia + pd.offsets.MonthEnd(1)
        dia = primeiro_dia
        while dia <= ultimo_dia:
            # Intervalo 0:00 - 11:59
            inicio1 = dia
            fim1 = dia + timedelta(hours=11, minutes=59)
            df1 = coletar_pedidos_intervalo(inicio1, fim1)
            if not df1.empty:
                df_ano = pd.concat([df_ano, df1], ignore_index=True)

            # Intervalo 12:00 - 23:59
            inicio2 = dia + timedelta(hours=12)
            fim2 = dia + timedelta(hours=23, minutes=59)
            df2 = coletar_pedidos_intervalo(inicio2, fim2)
            if not df2.empty:
                df_ano = pd.concat([df_ano, df2], ignore_index=True)

            dia += timedelta(days=1)
    return df_ano

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")

# EXECUÇÃO
if __name__ == "__main__":
    ANO = 2025
    if not testar_token():
        exit("❌ Corrija token ou permissões antes de continuar.")

    # Limpar tabela antes de gravar
    with engine.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS pedidos"))

    df_ano = coletar_ano_12h(ANO)
    print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
    armazenar_no_sql(df_ano, tabela="pedidos")
    print("🏁 Processo finalizado com sucesso!")


✅ Token válido
🔄 Coletando 2025-01-01 00:00:00 a 2025-01-01 11:59:00, página 1...
🔄 Coletando 2025-01-01 12:00:00 a 2025-01-01 23:59:00, página 1...
🔄 Coletando 2025-01-02 00:00:00 a 2025-01-02 11:59:00, página 1...
🔄 Coletando 2025-01-02 12:00:00 a 2025-01-02 23:59:00, página 1...
🔄 Coletando 2025-01-03 00:00:00 a 2025-01-03 11:59:00, página 1...
🔄 Coletando 2025-01-03 12:00:00 a 2025-01-03 23:59:00, página 1...
🔄 Coletando 2025-01-04 00:00:00 a 2025-01-04 11:59:00, página 1...
🔄 Coletando 2025-01-04 12:00:00 a 2025-01-04 23:59:00, página 1...
🔄 Coletando 2025-01-05 00:00:00 a 2025-01-05 11:59:00, página 1...
🔄 Coletando 2025-01-05 12:00:00 a 2025-01-05 23:59:00, página 1...
🔄 Coletando 2025-01-06 00:00:00 a 2025-01-06 11:59:00, página 1...
🔄 Coletando 2025-01-06 12:00:00 a 2025-01-06 23:59:00, página 1...
🔄 Coletando 2025-01-07 00:00:00 a 2025-01-07 11:59:00, página 1...
🔄 Coletando 2025-01-07 12:00:00 a 2025-01-07 23:59:00, página 1...
🔄 Coletando 2025-01-08 00:00:00 a 2025-01-08 11

In [34]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import timedelta, datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def coletar_dia(data: datetime):
    """Coleta todos os pedidos de um dia, paginando se necessário"""
    pagina = 1
    df_total = pd.DataFrame()
    limite = 5000

    while True:
        params = {
            "dataInicial": data.strftime("%Y-%m-%d"),
            "dataFinal": data.strftime("%Y-%m-%d"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} no dia {data.strftime('%Y-%m-%d')}, página {pagina}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Convertendo colunas complexas para JSON
        colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
        for col in colunas_complexas:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df["data"] = data
        df["pagina"] = pagina
        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.2)

    print(f"📦 Total de registros no dia {data.strftime('%Y-%m-%d')}: {len(df_total)}")
    return df_total

def coletar_mes(ano: int, mes: int, max_threads=5):
    """Coleta todos os dias do mês usando threads"""
    data_inicial = pd.to_datetime(f"{ano}-{mes:02d}-01")
    data_final = data_inicial + pd.offsets.MonthEnd(1)

    dias = pd.date_range(data_inicial, data_final)
    df_mes = pd.DataFrame()

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(coletar_dia, dia): dia for dia in dias}
        for future in as_completed(futures):
            df_dia = future.result()
            if not df_dia.empty:
                df_mes = pd.concat([df_mes, df_dia], ignore_index=True)

    print(f"📦 Total de registros coletados no mês {mes:02d}: {len(df_mes)}")
    return df_mes

def coletar_ano(ano: int, max_threads=5):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_mes(ano, mes, max_threads=max_threads)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")

# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    # Limpar tabela antes
    with engine.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS pedidos"))

    df_ano = coletar_ano(ANO, max_threads=10)  # aumenta threads se quiser mais velocidade
    print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
    armazenar_no_sql(df_ano, tabela="pedidos")
    print("🏁 Processo finalizado com sucesso!")


📦 Total de registros no dia 2025-01-05: 100
📦 Total de registros no dia 2025-01-07: 100
📦 Total de registros no dia 2025-01-03: 100
📦 Total de registros no dia 2025-01-06: 100
📦 Total de registros no dia 2025-01-10: 100
📦 Total de registros no dia 2025-01-08: 100
📦 Total de registros no dia 2025-01-01: 74
📦 Total de registros no dia 2025-01-04: 100
📦 Total de registros no dia 2025-01-09: 100
📦 Total de registros no dia 2025-01-02: 100
📦 Total de registros no dia 2025-01-11: 100
📦 Total de registros no dia 2025-01-12: 100
📦 Total de registros no dia 2025-01-14: 68
📦 Total de registros no dia 2025-01-15: 100
📦 Total de registros no dia 2025-01-13: 100
📦 Total de registros no dia 2025-01-18: 100
📦 Total de registros no dia 2025-01-19: 100
📦 Total de registros no dia 2025-01-16: 100
📦 Total de registros no dia 2025-01-17: 100
📦 Total de registros no dia 2025-01-20: 100
📦 Total de registros no dia 2025-01-21: 100
📦 Total de registros no dia 2025-01-22: 100
📦 Total de registros no dia 2025-0

ProgrammingError: (pyodbc.ProgrammingError) ('42S21', "[42S21] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Os nomes de colunas em cada tabela devem ser exclusivos. O nome da coluna 'data' na tabela 'pedidos' foi especificado mais de uma vez. (2705) (SQLExecDirectW)")
[SQL: 
CREATE TABLE pedidos (
	[ID] VARCHAR(max) NULL, 
	[AndroidVendaID] VARCHAR(max) NULL, 
	[Codigo] BIGINT NULL, 
	[OrigemVenda] VARCHAR(max) NULL, 
	[DepositoID] VARCHAR(max) NULL, 
	[Tabela] VARCHAR(max) NULL, 
	[TabelaID] VARCHAR(max) NULL, 
	[Deposito] VARCHAR(max) NULL, 
	[StatusSistema] VARCHAR(max) NULL, 
	[Status] VARCHAR(max) NULL, 
	[Categoria] VARCHAR(max) NULL, 
	[Validade] VARCHAR(max) NULL, 
	[Empresa] VARCHAR(max) NULL, 
	[EmpresaID] VARCHAR(max) NULL, 
	[ClienteID] VARCHAR(max) NULL, 
	[Cliente] VARCHAR(max) NULL, 
	[PessoaID] VARCHAR(max) NULL, 
	[ClienteCNPJ] VARCHAR(max) NULL, 
	[ClienteEmail] VARCHAR(max) NULL, 
	[Vendedor] VARCHAR(max) NULL, 
	[VendedorID] VARCHAR(max) NULL, 
	[PlanoDeCon

In [35]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================

def coletar_intervalo(inicio: datetime, fim: datetime):
    """Coleta todos os pedidos de um intervalo, paginando se necessário"""
    pagina = 1
    df_total = pd.DataFrame()
    limite = 5000

    while True:
        params = {
            "dataInicial": inicio.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": fim.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code}, intervalo {inicio} a {fim}, página {pagina}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Convertendo colunas complexas para JSON
        colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
        for col in colunas_complexas:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df["inicio_intervalo"] = inicio
        df["fim_intervalo"] = fim
        df["pagina"] = pagina
        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.2)

    print(f"📦 Total de registros no intervalo {inicio.strftime('%Y-%m-%d %H:%M')} a {fim.strftime('%H:%M')}: {len(df_total)}")
    return df_total

def coletar_dia_em_intervalos(dia: datetime, max_threads=2):
    """Divide o dia em dois intervalos de 12h e processa em paralelo"""
    intervalos = [
        (dia, dia + timedelta(hours=11, minutes=59, seconds=59)),
        (dia + timedelta(hours=12), dia + timedelta(hours=23, minutes=59, seconds=59))
    ]

    df_dia = pd.DataFrame()
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(coletar_intervalo, inicio, fim): (inicio, fim) for inicio, fim in intervalos}
        for future in as_completed(futures):
            df_intervalo = future.result()
            if not df_intervalo.empty:
                df_dia = pd.concat([df_dia, df_intervalo], ignore_index=True)

    return df_dia

def coletar_mes(ano: int, mes: int, max_threads_dia=2, max_threads_mes=5):
    """Coleta todos os dias do mês usando threads"""
    data_inicial = pd.to_datetime(f"{ano}-{mes:02d}-01")
    data_final = data_inicial + pd.offsets.MonthEnd(1)
    dias = pd.date_range(data_inicial, data_final)

    df_mes = pd.DataFrame()
    with ThreadPoolExecutor(max_workers=max_threads_mes) as executor:
        futures = {executor.submit(coletar_dia_em_intervalos, dia, max_threads=max_threads_dia): dia for dia in dias}
        for future in as_completed(futures):
            df_dia = future.result()
            if not df_dia.empty:
                df_mes = pd.concat([df_mes, df_dia], ignore_index=True)

    print(f"📦 Total de registros coletados no mês {mes:02d}: {len(df_mes)}")
    return df_mes

def coletar_ano(ano: int, max_threads_dia=2, max_threads_mes=5):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_mes(ano, mes, max_threads_dia=max_threads_dia, max_threads_mes=max_threads_mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")

# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    # Limpar tabela antes
    with engine.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS pedidos"))

    df_ano = coletar_ano(ANO, max_threads_dia=2, max_threads_mes=5)
    print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
    armazenar_no_sql(df_ano, tabela="pedidos")
    print("🏁 Processo finalizado com sucesso!")


📦 Total de registros no intervalo 2025-01-05 12:00 a 23:59: 42
📦 Total de registros no intervalo 2025-01-02 00:00 a 11:59: 100
📦 Total de registros no intervalo 2025-01-03 00:00 a 11:59: 100
📦 Total de registros no intervalo 2025-01-01 12:00 a 23:59: 16
📦 Total de registros no intervalo 2025-01-03 12:00 a 23:59: 100
📦 Total de registros no intervalo 2025-01-04 00:00 a 11:59: 100
📦 Total de registros no intervalo 2025-01-01 00:00 a 11:59: 74
📦 Total de registros no intervalo 2025-01-05 00:00 a 11:59: 100
📦 Total de registros no intervalo 2025-01-02 12:00 a 23:59: 100
📦 Total de registros no intervalo 2025-01-04 12:00 a 23:59: 100
📦 Total de registros no intervalo 2025-01-06 12:00 a 23:59: 83
📦 Total de registros no intervalo 2025-01-06 00:00 a 11:59: 100
📦 Total de registros no intervalo 2025-01-07 00:00 a 11:59: 100
📦 Total de registros no intervalo 2025-01-08 12:00 a 23:59: 92
📦 Total de registros no intervalo 2025-01-09 12:00 a 23:59: 94
📦 Total de registros no intervalo 2025-01-08 0

In [36]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import timedelta, datetime
from concurrent.futures import ThreadPoolExecutor

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    """Verifica se o token está válido"""
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido")
        return True
    else:
        print(f"❌ Token ou acesso inválido ({resp.status_code})")
        return False

def coletar_intervalo(inicio: datetime, fim: datetime):
    """Coleta pedidos de um intervalo específico de horário"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000

    while True:
        params = {
            "dataInicial": inicio.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": fim.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} no intervalo {inicio} a {fim}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Filtra apenas registros dentro do intervalo exato
        df["DataFaturamentoPedido"] = pd.to_datetime(df["DataFaturamentoPedido"])
        df = df[(df["DataFaturamentoPedido"] >= inicio) & (df["DataFaturamentoPedido"] <= fim)]

        # Colunas complexas em JSON
        colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
        for col in colunas_complexas:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.2)

    return df_total

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês dividindo cada dia em 2 intervalos"""
    data_inicial = pd.to_datetime(f"{ano}-{mes:02d}-01")
    data_final = data_inicial + pd.offsets.MonthEnd(1)

    df_total_mes = pd.DataFrame()
    intervalos = []

    # Cria intervalos de 12h por dia
    dia_atual = data_inicial
    while dia_atual <= data_final:
        intervalo1 = (dia_atual, dia_atual + timedelta(hours=11, minutes=59, seconds=59))
        intervalo2 = (dia_atual + timedelta(hours=12), dia_atual + timedelta(hours=23, minutes=59, seconds=59))
        intervalos.append(intervalo1)
        intervalos.append(intervalo2)
        dia_atual += timedelta(days=1)

    # Coleta em paralelo
    with ThreadPoolExecutor(max_workers=4) as executor:
        resultados = list(executor.map(lambda x: coletar_intervalo(x[0], x[1]), intervalos))

    # Concatena resultados
    for df in resultados:
        if not df.empty:
            df_total_mes = pd.concat([df_total_mes, df], ignore_index=True)

    # Remove duplicatas
    df_total_mes = df_total_mes.drop_duplicates(subset=["ID"])

    print(f"📦 Total de registros coletados no mês {mes:02d}: {len(df_total_mes)}")
    return df_total_mes

def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")

# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token ou permissões antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        # Coleta do ano inteiro
        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")

        # Armazena no SQL Server
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


✅ Token válido


KeyError: 'DataFaturamentoPedido'

In [37]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido, podemos coletar os dados!")
        return True
    else:
        print(f"❌ Token ou acesso inválido ({resp.status_code})")
        return False


def coletar_intervalo(inicio: datetime, fim: datetime):
    """Coleta pedidos de um intervalo específico de horário"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000

    while True:
        params = {
            "dataInicial": inicio.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": fim.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} no intervalo {inicio} a {fim}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Garantir que a coluna exista antes de filtrar
        if "DataFaturamentoPedido" in df.columns:
            df["DataFaturamentoPedido"] = pd.to_datetime(df["DataFaturamentoPedido"])
            df = df[(df["DataFaturamentoPedido"] >= inicio) & (df["DataFaturamentoPedido"] <= fim)]
        else:
            df["DataFaturamentoPedido"] = pd.NaT  # cria a coluna vazia

        # Colunas complexas em JSON
        colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
        for col in colunas_complexas:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.2)

    return df_total


def coletar_pedidos_dia(dia: datetime):
    """Divide o dia em dois intervalos de 12h"""
    intervalo = timedelta(hours=12)
    inicios = [dia, dia + intervalo]
    resultados = []

    for inicio in inicios:
        fim = min(inicio + intervalo - timedelta(seconds=1), dia + timedelta(days=1) - timedelta(seconds=1))
        df = coletar_intervalo(inicio, fim)
        if not df.empty:
            resultados.append(df)
    if resultados:
        return pd.concat(resultados, ignore_index=True)
    return pd.DataFrame()


def coletar_ano(ano: int):
    dias = pd.date_range(start=f"{ano}-01-01", end=f"{ano}-12-31")
    df_total = pd.DataFrame()

    with ThreadPoolExecutor(max_workers=8) as executor:
        resultados = list(executor.map(coletar_pedidos_dia, dias))

    for df in resultados:
        if not df.empty:
            df_total = pd.concat([df_total, df], ignore_index=True)

    return df_total


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token ou permissões antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        # Coletar todos os dias do ano
        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")

        # Armazenar no SQL Server
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


✅ Token válido, podemos coletar os dados!

📊 Total de registros coletados no ano 2025: 50738
💾 Gravando 50738 registros na tabela 'pedidos'...
✅ Dados gravados com sucesso
🏁 Processo finalizado com sucesso!


In [38]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import timedelta, datetime

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido, podemos coletar os dados!")
        return True
    else:
        print(f"❌ Token ou acesso inválido ({resp.status_code})")
        return False

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês, dividindo por dias, com deduplicação"""
    data_inicial = pd.to_datetime(f"{ano}-{mes:02d}-01")
    data_final = data_inicial + pd.offsets.MonthEnd(1)
    df_total = pd.DataFrame()

    intervalo = timedelta(days=1)
    inicio = data_inicial

    while inicio <= data_final:
        fim = min(inicio + intervalo - timedelta(seconds=1), data_final)
        pagina = 1

        while True:
            params = {
                "dataInicial": inicio.strftime("%Y-%m-%d"),
                "dataFinal": fim.strftime("%Y-%m-%d"),
                "filtrarPor": "DataFaturamentoPedido",
                "empresa": EMPRESA,
                "pagina": pagina,
                "limite": 5000
            }

            print(f"🔄 Coletando {inicio.strftime('%Y-%m-%d')} a {fim.strftime('%Y-%m-%d')}, página {pagina}...")
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code != 200:
                print(f"⚠️ Erro {resp.status_code}, página {pagina}")
                break

            dados = resp.json()
            if isinstance(dados, dict):
                df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                df = pd.DataFrame()

            if df.empty:
                break

            # Colunas complexas em JSON para manter recolhidas
            colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
            for col in colunas_complexas:
                df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina
            df_total = pd.concat([df_total, df], ignore_index=True)

            if len(df) < 5000:
                break
            pagina += 1
            time.sleep(0.2)

        inicio = fim + timedelta(seconds=1)

    print(f"📦 Total de registros coletados no mês {mes:02d}: {len(df_total)}")
    return df_total

def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)

    # Deduplicar pelo ID do pedido
    if "ID" in df_ano.columns:
        df_ano = df_ano.drop_duplicates(subset="ID")
    elif "Codigo" in df_ano.columns:
        df_ano = df_ano.drop_duplicates(subset="Codigo")

    print(f"📊 Total de registros únicos no ano {ano}: {len(df_ano)}")
    return df_ano

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")

# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token ou permissões antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        # Coletar todos os meses e deduplicar
        df_ano = coletar_ano(ANO)
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


✅ Token válido, podemos coletar os dados!
🔄 Coletando 2025-01-01 a 2025-01-01, página 1...
🔄 Coletando 2025-01-02 a 2025-01-02, página 1...
🔄 Coletando 2025-01-03 a 2025-01-03, página 1...
🔄 Coletando 2025-01-04 a 2025-01-04, página 1...
🔄 Coletando 2025-01-05 a 2025-01-05, página 1...
🔄 Coletando 2025-01-06 a 2025-01-06, página 1...
🔄 Coletando 2025-01-07 a 2025-01-07, página 1...
🔄 Coletando 2025-01-08 a 2025-01-08, página 1...
🔄 Coletando 2025-01-09 a 2025-01-09, página 1...
🔄 Coletando 2025-01-10 a 2025-01-10, página 1...
🔄 Coletando 2025-01-11 a 2025-01-11, página 1...
🔄 Coletando 2025-01-12 a 2025-01-12, página 1...
🔄 Coletando 2025-01-13 a 2025-01-13, página 1...
🔄 Coletando 2025-01-14 a 2025-01-14, página 1...
🔄 Coletando 2025-01-15 a 2025-01-15, página 1...
🔄 Coletando 2025-01-16 a 2025-01-16, página 1...
🔄 Coletando 2025-01-17 a 2025-01-17, página 1...
🔄 Coletando 2025-01-18 a 2025-01-18, página 1...
🔄 Coletando 2025-01-19 a 2025-01-19, página 1...
🔄 Coletando 2025-01-20 a 20

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

In [39]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import timedelta, datetime

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    """Verifica se o token está válido"""
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido")
        return True
    else:
        print(f"❌ Token ou acesso inválido ({resp.status_code})")
        return False


def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta pedidos de um mês, dividindo por dias com overlap de 1 minuto"""
    df_total = pd.DataFrame()
    data_inicial = pd.to_datetime(f"{ano}-{mes:02d}-01")
    data_final = data_inicial + pd.offsets.MonthEnd(1)
    
    dia = data_inicial
    while dia <= data_final:
        # Intervalo do dia com overlap de 1 minuto
        inicio = dia
        fim = dia + timedelta(days=1) - timedelta(seconds=60)
        pagina = 1

        while True:
            params = {
                "dataInicial": inicio.strftime("%Y-%m-%dT%H:%M:%S"),
                "dataFinal": fim.strftime("%Y-%m-%dT%H:%M:%S"),
                "filtrarPor": "DataFaturamentoPedido",
                "empresa": EMPRESA,
                "pagina": pagina,
                "limite": 5000
            }

            print(f"🔄 Coletando {inicio} a {fim}, página {pagina}...")
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)

            if resp.status_code != 200:
                print(f"⚠️ Erro {resp.status_code} na página {pagina}")
                break

            dados = resp.json()
            if isinstance(dados, dict):
                df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                df = pd.DataFrame()

            if df.empty:
                break

            # Colunas complexas ficam JSON para expandir depois
            colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
            for col in colunas_complexas:
                df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

            df["ano"] = ano
            df["mes"] = mes
            df["pagina"] = pagina

            df_total = pd.concat([df_total, df], ignore_index=True)

            if len(df) < 5000:
                break
            pagina += 1
            time.sleep(0.2)

        dia += timedelta(days=1)

    # Deduplicar usando ID ou Codigo do pedido
    if "ID" in df_total.columns:
        df_total = df_total.drop_duplicates(subset="ID")
    elif "Codigo" in df_total.columns:
        df_total = df_total.drop_duplicates(subset="Codigo")

    print(f"📦 Total de registros coletados no mês {mes:02d}: {len(df_total)}")
    return df_total


def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


✅ Token válido
🔄 Coletando 2025-01-01 00:00:00 a 2025-01-01 23:59:00, página 1...
🔄 Coletando 2025-01-02 00:00:00 a 2025-01-02 23:59:00, página 1...
🔄 Coletando 2025-01-03 00:00:00 a 2025-01-03 23:59:00, página 1...
🔄 Coletando 2025-01-04 00:00:00 a 2025-01-04 23:59:00, página 1...
🔄 Coletando 2025-01-05 00:00:00 a 2025-01-05 23:59:00, página 1...
🔄 Coletando 2025-01-06 00:00:00 a 2025-01-06 23:59:00, página 1...
🔄 Coletando 2025-01-07 00:00:00 a 2025-01-07 23:59:00, página 1...
🔄 Coletando 2025-01-08 00:00:00 a 2025-01-08 23:59:00, página 1...
🔄 Coletando 2025-01-09 00:00:00 a 2025-01-09 23:59:00, página 1...
🔄 Coletando 2025-01-10 00:00:00 a 2025-01-10 23:59:00, página 1...
🔄 Coletando 2025-01-11 00:00:00 a 2025-01-11 23:59:00, página 1...
🔄 Coletando 2025-01-12 00:00:00 a 2025-01-12 23:59:00, página 1...
🔄 Coletando 2025-01-13 00:00:00 a 2025-01-13 23:59:00, página 1...
🔄 Coletando 2025-01-14 00:00:00 a 2025-01-14 23:59:00, página 1...
🔄 Coletando 2025-01-15 00:00:00 a 2025-01-15 23

In [40]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm  # barra de progresso

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido, podemos coletar os dados!")
        return True
    else:
        print(f"❌ Token ou acesso inválido ({resp.status_code})")
        return False

def coletar_intervalo(inicio: datetime, fim: datetime):
    """Coleta todos os pedidos de um intervalo específico, paginando até o limite."""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000

    while True:
        params = {
            "dataInicial": inicio.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": fim.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        try:
            resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)
            if resp.status_code != 200:
                print(f"⚠️ Erro {resp.status_code} no intervalo {inicio} a {fim}")
                break

            dados = resp.json()
            if isinstance(dados, dict):
                df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
            elif isinstance(dados, list):
                df = pd.DataFrame(dados)
            else:
                df = pd.DataFrame()

            if df.empty:
                break

            # Colunas complexas em JSON
            colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
            for col in colunas_complexas:
                df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

            df_total = pd.concat([df_total, df], ignore_index=True)

            if len(df) < limite:
                break
            pagina += 1
            time.sleep(0.2)
        except Exception as e:
            print(f"⚠️ Erro no intervalo {inicio} a {fim}: {e}")
            break

    return df_total

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos do mês usando threads por intervalos de 30 minutos."""
    data_inicial = datetime(ano, mes, 1)
    data_final = data_inicial + pd.offsets.MonthEnd(1)
    intervalos = []

    atual = data_inicial
    while atual <= data_final:
        fim = min(atual + timedelta(minutes=30) - timedelta(seconds=1), data_final)
        intervalos.append((atual, fim))
        atual = fim + timedelta(seconds=1)

    df_total = pd.DataFrame()
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(coletar_intervalo, start, end): (start, end) for start, end in intervalos}
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"Mês {mes:02d}"):
            df = future.result()
            if not df.empty:
                df_total = pd.concat([df_total, df], ignore_index=True)

    # Deduplicar por ID
    if "ID" in df_total.columns:
        df_total.drop_duplicates(subset="ID", inplace=True)

    print(f"📦 Total de registros coletados no mês {mes:02d}: {len(df_total)}")
    return df_total

def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="replace", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")

# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token ou permissões antes de continuar.")
    else:
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


ModuleNotFoundError: No module named 'tqdm'

In [42]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    """Verifica se o token está válido"""
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido!")
        return True
    else:
        print(f"❌ Token inválido ou acesso negado ({resp.status_code})")
        return False

def coletar_intervalo(inicio: datetime, fim: datetime):
    """Coleta dados de um intervalo de datas, paginando"""
    pagina = 1
    df_total = pd.DataFrame()

    while True:
        params = {
            "dataInicial": inicio.strftime("%Y-%m-%d"),
            "dataFinal": fim.strftime("%Y-%m-%d"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": 5000
        }

        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} no intervalo {inicio} a {fim}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Manter colunas complexas como JSON
        colunas_complexas = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, (list, dict))).any()]
        for c in colunas_complexas:
            df[c] = df[c].apply(lambda x: json.dumps(x) if x is not None else None)

        df["inicio_intervalo"] = inicio
        df["fim_intervalo"] = fim
        df["pagina"] = pagina
        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < 5000:
            break
        pagina += 1
        time.sleep(0.2)

    return df_total

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os dias do mês, divididos em dois intervalos de 12h"""
    df_mes = pd.DataFrame()
    data_inicial = datetime(ano, mes, 1)
    data_final = data_inicial + pd.offsets.MonthEnd(1)
    dias = [data_inicial + timedelta(days=i) for i in range((data_final - data_inicial).days + 1)]

    # ThreadPool para cada dia
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for dia in dias:
            # Intervalo 00:00-11:59
            inicio1 = dia
            fim1 = dia + timedelta(hours=11, minutes=59)
            futures.append(executor.submit(coletar_intervalo, inicio1, fim1))
            # Intervalo 12:00-23:59
            inicio2 = dia + timedelta(hours=12)
            fim2 = dia + timedelta(hours=23, minutes=59)
            futures.append(executor.submit(coletar_intervalo, inicio2, fim2))

        for future in as_completed(futures):
            df_total = future.result()
            if not df_total.empty:
                df_mes = pd.concat([df_mes, df_total], ignore_index=True)

    # Remover duplicados caso existam
    df_mes.drop_duplicates(inplace=True)
    print(f"📦 Total de registros no mês {mes:02d}: {len(df_mes)}")
    return df_mes

def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    df_ano.drop_duplicates(inplace=True)
    return df_ano

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")

# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token antes de continuar")
    else:
        # Limpar tabela antes
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado!")

✅ Token válido!
📦 Total de registros no mês 01: 6084
📦 Total de registros no mês 02: 5566
📦 Total de registros no mês 03: 6126
📦 Total de registros no mês 04: 5710
📦 Total de registros no mês 05: 6120
📦 Total de registros no mês 06: 5968
📦 Total de registros no mês 07: 6200
📦 Total de registros no mês 08: 6140
📦 Total de registros no mês 09: 6000
📦 Total de registros no mês 10: 2400
📦 Total de registros no mês 11: 0
📦 Total de registros no mês 12: 0

📊 Total de registros coletados no ano 2025: 56314
💾 Gravando 56314 registros na tabela 'pedidos'...
✅ Dados gravados com sucesso
🏁 Processo finalizado!


In [43]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import json
import time
from datetime import datetime
from tqdm import tqdm  # barra de progresso opcional

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    """Testa se o token está válido"""
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido!")
        return True
    else:
        print(f"❌ Token inválido ou acesso negado ({resp.status_code})")
        return False

def coletar_pedidos_mes(ano: int, mes: int):
    """Coleta todos os pedidos de um mês inteiro com paginação"""
    data_inicial = f"{ano}-{mes:02d}-01"
    data_final = (pd.to_datetime(data_inicial) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")
    df_total = pd.DataFrame()
    pagina = 1
    limite = 5000  # máximo permitido

    while True:
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        print(f"🔄 Coletando {data_inicial} a {data_final}, página {pagina}...")
        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code}, interrompendo o mês {mes}")
            break

        dados = resp.json()

        # Detecta se o retorno é dict ou list
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Transformar colunas complexas em JSON
        colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
        for col in colunas_complexas:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < limite:  # última página
            break
        pagina += 1
        time.sleep(0.2)  # evitar sobrecarga da API

    # Remove duplicados (caso API retorne repetidos)
    if "ID" in df_total.columns:
        df_total.drop_duplicates(subset="ID", inplace=True)

    print(f"📦 Total de registros no mês {mes}: {len(df_total)}")
    return df_total

def coletar_ano(ano: int):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print("✅ Dados gravados com sucesso")

# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        # Coletar todos os meses do ano
        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


ModuleNotFoundError: No module named 'tqdm'

In [46]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================
# CONFIGURAÇÕES
# =====================================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# Conexão com SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# =====================================================
# FUNÇÕES
# =====================================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido, podemos coletar os dados!")
        return True
    else:
        print(f"❌ Token ou acesso inválido ({resp.status_code})")
        return False


def coletar_intervalo(inicio, fim):
    """Coleta todos os pedidos de um intervalo de datas"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 10000  # máximo permitido pela API

    while True:
        params = {
            "dataInicial": inicio.strftime("%Y-%m-%d"),
            "dataFinal": fim.strftime("%Y-%m-%d"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }

        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=60)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} para {inicio} a {fim}, página {pagina}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Converter colunas complexas para JSON (ficam recolhidas)
        colunas_complexas = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (list, dict))).any()]
        for col in colunas_complexas:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df["dataInicio"] = inicio
        df["dataFim"] = fim
        df["pagina"] = pagina
        df_total = pd.concat([df_total, df], ignore_index=True)

        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.2)

    return df_total


def coletar_pedidos_mes(ano, mes):
    """Divide o mês em dias e coleta pedidos em paralelo"""
    data_inicial = pd.to_datetime(f"{ano}-{mes:02d}-01")
    data_final = data_inicial + pd.offsets.MonthEnd(1)
    dias = pd.date_range(data_inicial, data_final, freq="D")

    df_mes = pd.DataFrame()
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for dia in dias:
            # cada dia pode ser dividido em duas metades se quiser
            inicio1 = dia
            fim1 = dia
            futures.append(executor.submit(coletar_intervalo, inicio1, fim1))

        for future in as_completed(futures):
            df_dia = future.result()
            if not df_dia.empty:
                df_mes = pd.concat([df_mes, df_dia], ignore_index=True)

    print(f"📦 Total de registros coletados no mês {mes:02d}: {len(df_mes)}")
    return df_mes


def coletar_ano(ano):
    df_ano = pd.DataFrame()
    for mes in range(1, 13):
        df_mes = coletar_pedidos_mes(ano, mes)
        if not df_mes.empty:
            df_ano = pd.concat([df_ano, df_mes], ignore_index=True)
    return df_ano


def armazenar_no_sql(df, tabela):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    print(f"💾 Gravando {len(df)} registros na tabela '{tabela}'...")
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=1000)
    print("✅ Dados gravados com sucesso")


# =====================================================
# EXECUÇÃO
# =====================================================
if __name__ == "__main__":
    ANO = 2025

    if not testar_token():
        print("❌ Corrija o token ou permissões antes de continuar.")
    else:
        # Limpar tabela antes de gravar
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        df_ano = coletar_ano(ANO)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")
        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado com sucesso!")


✅ Token válido, podemos coletar os dados!
📦 Total de registros coletados no mês 01: 3042
📦 Total de registros coletados no mês 02: 2783
📦 Total de registros coletados no mês 03: 3063
📦 Total de registros coletados no mês 04: 2855
📦 Total de registros coletados no mês 05: 3060
📦 Total de registros coletados no mês 06: 2984
📦 Total de registros coletados no mês 07: 3100
📦 Total de registros coletados no mês 08: 3070
📦 Total de registros coletados no mês 09: 3000
📦 Total de registros coletados no mês 10: 1200
📦 Total de registros coletados no mês 11: 0
📦 Total de registros coletados no mês 12: 0

📊 Total de registros coletados no ano 2025: 28157
💾 Gravando 28157 registros na tabela 'pedidos'...
✅ Dados gravados com sucesso
🏁 Processo finalizado com sucesso!


In [1]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================================
# CONFIGURAÇÕES
# ================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = "CASA DE SUCO - BARRA DO CORDA"

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# ================================
# FUNÇÕES
# ================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido!")
        return True
    else:
        print(f"❌ Token inválido ({resp.status_code})")
        return False

def coletar_pedidos_intervalo(start: datetime, end: datetime):
    """Coleta pedidos de um intervalo de tempo específico com paginação"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 1000

    while True:
        params = {
            "dataInicial": start.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": end.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }
        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=180)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} em {start} -> {end}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Colunas complexas
        col_complex = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (dict,list))).any()]
        for col in col_complex:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df_total = pd.concat([df_total, df], ignore_index=True)
        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.1)

    return df_total

def coletar_pedidos_dia(dia: datetime):
    """Divide o dia em intervalos de 1 hora"""
    df_dia = pd.DataFrame()
    hora_inicio = datetime(dia.year, dia.month, dia.day, 0, 0, 0)
    while hora_inicio < datetime(dia.year, dia.month, dia.day, 23, 59, 59):
        hora_fim = min(hora_inicio + timedelta(hours=1), datetime(dia.year, dia.month, dia.day, 23, 59, 59))
        df_intervalo = coletar_pedidos_intervalo(hora_inicio, hora_fim)
        if not df_intervalo.empty:
            df_dia = pd.concat([df_dia, df_intervalo], ignore_index=True)
        hora_inicio = hora_fim + timedelta(seconds=1)

    if "ID" in df_dia.columns:
        df_dia = df_dia.drop_duplicates(subset=["ID"])
    print(f"📦 Total de registros no dia {dia.strftime('%Y-%m-%d')}: {len(df_dia)}")
    return df_dia

def coletar_ano_multithread(ano: int, max_workers: int = 5):
    """Coleta todo o ano usando múltiplas threads"""
    dias = [datetime(ano, 1, 1) + timedelta(days=i) for i in range(0, 365)]
    df_ano_total = pd.DataFrame()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(coletar_pedidos_dia, dia): dia for dia in dias}
        for future in as_completed(futures):
            df_dia = future.result()
            if not df_dia.empty:
                df_ano_total = pd.concat([df_ano_total, df_dia], ignore_index=True)

    if "ID" in df_ano_total.columns:
        df_ano_total = df_ano_total.drop_duplicates(subset=["ID"])
    return df_ano_total

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print(f"💾 Gravados {len(df)} registros na tabela {tabela}!")

# ================================
# EXECUÇÃO
# ================================
if __name__ == "__main__":
    ANO = 2025
    MAX_THREADS = 10  # ajuste conforme sua máquina

    if not testar_token():
        print("❌ Corrija o token antes de continuar")
    else:
        # Limpar tabela antes
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        # Coleta do ano inteiro com threads
        df_ano = coletar_ano_multithread(ANO, max_workers=MAX_THREADS)
        print(f"\n📊 Total de registros coletados no ano {ANO}: {len(df_ano)}")

        armazenar_no_sql(df_ano, tabela="pedidos")
        print("🏁 Processo finalizado!")

✅ Token válido!
📦 Total de registros no dia 2025-01-01: 74
📦 Total de registros no dia 2025-01-05: 152
📦 Total de registros no dia 2025-01-08: 143
📦 Total de registros no dia 2025-01-06: 117
📦 Total de registros no dia 2025-01-10: 121
📦 Total de registros no dia 2025-01-03: 206
📦 Total de registros no dia 2025-01-09: 137
📦 Total de registros no dia 2025-01-04: 153
📦 Total de registros no dia 2025-01-07: 141
📦 Total de registros no dia 2025-01-02: 159
📦 Total de registros no dia 2025-01-12: 115
📦 Total de registros no dia 2025-01-11: 164
📦 Total de registros no dia 2025-01-14: 68
📦 Total de registros no dia 2025-01-15: 119
📦 Total de registros no dia 2025-01-13: 142
📦 Total de registros no dia 2025-01-16: 104
📦 Total de registros no dia 2025-01-19: 110
📦 Total de registros no dia 2025-01-17: 150
📦 Total de registros no dia 2025-01-18: 160
📦 Total de registros no dia 2025-01-20: 100
📦 Total de registros no dia 2025-01-21: 139
📦 Total de registros no dia 2025-01-22: 117
📦 Total de registr

In [1]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from calendar import monthrange

# ================================
# CONFIGURAÇÕES
# ================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESA = ["CASA DE SUCO - BARRA DO CORDA", "EMPORIO MIX"]

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# ================================
# FUNÇÕES
# ================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESA,
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido!")
        return True
    else:
        print(f"❌ Token inválido ({resp.status_code})")
        return False

def verificar_ou_criar_tabela():
    """Verifica se a tabela existe, se não existir cria uma básica"""
    query_verifica = """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'pedidos')
    BEGIN
        CREATE TABLE pedidos (
            ID NVARCHAR(255) PRIMARY KEY
        )
        PRINT '✅ Tabela pedidos criada'
    END
    ELSE
    BEGIN
        PRINT '✅ Tabela pedidos já existe'
    END
    """
    with engine.begin() as conn:
        conn.execute(text(query_verifica))

def coletar_pedidos_intervalo(start: datetime, end: datetime):
    """Coleta pedidos de um intervalo de tempo específico com paginação"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 1000

    while True:
        params = {
            "dataInicial": start.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": end.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": EMPRESA,
            "pagina": pagina,
            "limite": limite
        }
        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=180)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} em {start} -> {end}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        # Colunas complexas
        col_complex = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (dict,list))).any()]
        for col in col_complex:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df_total = pd.concat([df_total, df], ignore_index=True)
        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.1)

    return df_total

def coletar_pedidos_dia(dia: datetime):
    """Divide o dia em intervalos de 1 hora"""
    df_dia = pd.DataFrame()
    hora_inicio = datetime(dia.year, dia.month, dia.day, 0, 0, 0)
    while hora_inicio < datetime(dia.year, dia.month, dia.day, 23, 59, 59):
        hora_fim = min(hora_inicio + timedelta(hours=1), datetime(dia.year, dia.month, dia.day, 23, 59, 59))
        df_intervalo = coletar_pedidos_intervalo(hora_inicio, hora_fim)
        if not df_intervalo.empty:
            df_dia = pd.concat([df_dia, df_intervalo], ignore_index=True)
        hora_inicio = hora_fim + timedelta(seconds=1)

    if "ID" in df_dia.columns:
        df_dia = df_dia.drop_duplicates(subset=["ID"])
    print(f"📦 Total de registros no dia {dia.strftime('%Y-%m-%d')}: {len(df_dia)}")
    return df_dia

def coletar_mes_multithread(ano: int, mes: int, max_workers: int = 5):
    """Coleta apenas o mês especificado usando múltiplas threads"""
    # Descobre quantos dias tem o mês
    ultimo_dia = monthrange(ano, mes)[1]
    
    # Cria lista de datas apenas do mês especificado
    dias = [datetime(ano, mes, dia) for dia in range(1, ultimo_dia + 1)]
    df_mes_total = pd.DataFrame()

    print(f"📅 Coletando {len(dias)} dias do mês {mes:02d}/{ano}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(coletar_pedidos_dia, dia): dia for dia in dias}
        for future in as_completed(futures):
            df_dia = future.result()
            if not df_dia.empty:
                df_mes_total = pd.concat([df_mes_total, df_dia], ignore_index=True)

    if "ID" in df_mes_total.columns:
        df_mes_total = df_mes_total.drop_duplicates(subset=["ID"])
    return df_mes_total

def armazenar_no_sql_sem_duplicar(df: pd.DataFrame, tabela: str):
    """Armazena dados evitando duplicatas - insere apenas registros novos"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    
    # Remove colunas duplicadas se houver
    df = df.loc[:, ~df.columns.duplicated()]
    
    # 1. Criar tabela temporária com os dados novos
    tabela_temp = f"{tabela}_temp"
    df.to_sql(tabela_temp, con=engine, if_exists="replace", index=False, chunksize=500)
    print(f"📥 {len(df)} registros carregados na tabela temporária")
    
    # 2. Fazer MERGE para inserir apenas registros que não existem
    # Prepara a lista de colunas para o INSERT
    colunas = list(df.columns)
    colunas_insert = ', '.join([f'[{col}]' for col in colunas])
    colunas_values = ', '.join([f'source.[{col}]' for col in colunas])
    
    # Prepara UPDATE (todas as colunas exceto ID)
    colunas_update = [col for col in colunas if col != 'ID']
    update_set = ', '.join([f'target.[{col}] = source.[{col}]' for col in colunas_update])
    
    query_merge = f"""
    MERGE INTO [{tabela}] AS target
    USING [{tabela_temp}] AS source
    ON target.[ID] = source.[ID]
    WHEN NOT MATCHED THEN
        INSERT ({colunas_insert})
        VALUES ({colunas_values})
    """
    
    # Se houver colunas para atualizar, adiciona o WHEN MATCHED
    if colunas_update:
        query_merge += f"""
    WHEN MATCHED THEN
        UPDATE SET {update_set}
        """
    
    query_merge += ";"
    
    try:
        with engine.begin() as conn:
            # Executa o MERGE
            result = conn.execute(text(query_merge))
            rows_affected = result.rowcount
            
            # Remove a tabela temporária
            conn.execute(text(f"DROP TABLE [{tabela_temp}]"))
            
        print(f"💾 Sincronização concluída: {rows_affected} registros inseridos/atualizados na tabela {tabela}!")
        
    except Exception as e:
        print(f"❌ Erro ao fazer MERGE: {e}")
        # Em caso de erro, tenta limpar a tabela temporária
        try:
            with engine.begin() as conn:
                conn.execute(text(f"DROP TABLE IF EXISTS [{tabela_temp}]"))
        except:
            pass

# ================================
# EXECUÇÃO - MÊS ATUAL
# ================================
if __name__ == "__main__":
    # Detecta automaticamente mês e ano atuais
    hoje = datetime.now()
    ANO_ATUAL = hoje.year
    MES_ATUAL = hoje.month
    MAX_THREADS = 10

    print(f"🚀 Iniciando coleta do mês atual: {MES_ATUAL:02d}/{ANO_ATUAL}")
    print(f"📅 Data de hoje: {hoje.strftime('%d/%m/%Y')}")

    if not testar_token():
        print("❌ Corrija o token antes de continuar")
    else:
        # Verifica/cria a tabela (NÃO apaga os dados antigos)
        verificar_ou_criar_tabela()

        # Coleta do mês atual com threads
        df_mes = coletar_mes_multithread(ANO_ATUAL, MES_ATUAL, max_workers=MAX_THREADS)
        print(f"\n📊 Total de registros coletados no mês {MES_ATUAL:02d}/{ANO_ATUAL}: {len(df_mes)}")

        # Armazena sem duplicar
        armazenar_no_sql_sem_duplicar(df_mes, tabela="pedidos")
        print("🏁 Processo finalizado!")

🚀 Iniciando coleta do mês atual: 03/2026
📅 Data de hoje: 08/03/2026
✅ Token válido!
📅 Coletando 31 dias do mês 03/2026
📦 Total de registros no dia 2026-03-10: 0
📦 Total de registros no dia 2026-03-09: 0
📦 Total de registros no dia 2026-03-08: 152
📦 Total de registros no dia 2026-03-01: 234
📦 Total de registros no dia 2026-03-02: 175
📦 Total de registros no dia 2026-03-03: 131
📦 Total de registros no dia 2026-03-05: 164
📦 Total de registros no dia 2026-03-06: 177
📦 Total de registros no dia 2026-03-04: 135
📦 Total de registros no dia 2026-03-07: 195
📦 Total de registros no dia 2026-03-12: 0
📦 Total de registros no dia 2026-03-11: 0
📦 Total de registros no dia 2026-03-13: 0
📦 Total de registros no dia 2026-03-14: 0
📦 Total de registros no dia 2026-03-16: 0
📦 Total de registros no dia 2026-03-15: 0
📦 Total de registros no dia 2026-03-17: 0
📦 Total de registros no dia 2026-03-18: 0
📦 Total de registros no dia 2026-03-20: 0
📦 Total de registros no dia 2026-03-19: 0
📦 Total de registros no d

In [1]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from calendar import monthrange

# ================================
# CONFIGURAÇÕES
# ================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESAS = ["CASA DE SUCO - BARRA DO CORDA", "EMPORIO MIX"]

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_dados;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# ================================
# FUNÇÕES
# ================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESAS[0],
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido!")
        return True
    else:
        print(f"❌ Token inválido ({resp.status_code})")
        return False

def criar_tabela_se_nao_existe():
    """Cria a tabela pedidos se ela não existir"""
    query_criar = """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'pedidos')
    BEGIN
        CREATE TABLE pedidos (
            ID NVARCHAR(255) PRIMARY KEY,
            Empresa NVARCHAR(255),
            -- Adicione aqui outras colunas conforme necessário
            -- Exemplo: DataFaturamento DATETIME, ValorTotal DECIMAL(18,2), etc.
        )
    END
    """
    with engine.begin() as conn:
        conn.execute(text(query_criar))
    print("✅ Tabela 'pedidos' verificada/criada")

def coletar_pedidos_intervalo(start: datetime, end: datetime, empresa: str):
    """Coleta pedidos de um intervalo de tempo específico com paginação"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 1000

    while True:
        params = {
            "dataInicial": start.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": end.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": empresa,
            "pagina": pagina,
            "limite": limite
        }
        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=180)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} em {start} -> {end} para empresa {empresa}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        col_complex = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (dict,list))).any()]
        for col in col_complex:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df_total = pd.concat([df_total, df], ignore_index=True)
        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.1)

    return df_total

def coletar_pedidos_dia(dia: datetime, empresa: str):
    """Divide o dia em intervalos de 1 hora"""
    df_dia = pd.DataFrame()
    hora_inicio = datetime(dia.year, dia.month, dia.day, 0, 0, 0)
    while hora_inicio < datetime(dia.year, dia.month, dia.day, 23, 59, 59):
        hora_fim = min(hora_inicio + timedelta(hours=1), datetime(dia.year, dia.month, dia.day, 23, 59, 59))
        df_intervalo = coletar_pedidos_intervalo(hora_inicio, hora_fim, empresa)
        if not df_intervalo.empty:
            df_dia = pd.concat([df_dia, df_intervalo], ignore_index=True)
        hora_inicio = hora_fim + timedelta(seconds=1)

    if "ID" in df_dia.columns:
        df_dia = df_dia.drop_duplicates(subset=["ID"])
    print(f"📦 Total de registros no dia {dia.strftime('%Y-%m-%d')} para empresa {empresa}: {len(df_dia)}")
    return df_dia

def coletar_mes_atual_multithread(ano: int, mes: int, empresa: str, max_workers: int = 5):
    """Coleta apenas o mês especificado usando múltiplas threads"""
    ultimo_dia = monthrange(ano, mes)[1]
    dias = [datetime(ano, mes, dia) for dia in range(1, ultimo_dia + 1)]
    df_mes_total = pd.DataFrame()

    print(f"📅 Coletando dados de {len(dias)} dias do mês {mes}/{ano} para empresa {empresa}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(coletar_pedidos_dia, dia, empresa): dia for dia in dias}
        for future in as_completed(futures):
            df_dia = future.result()
            if not df_dia.empty:
                df_mes_total = pd.concat([df_mes_total, df_dia], ignore_index=True)

    if "ID" in df_mes_total.columns:
        df_mes_total = df_mes_total.drop_duplicates(subset=["ID"])
    return df_mes_total

def armazenar_no_sql_incremental(df: pd.DataFrame, tabela: str):
    """Armazena dados evitando duplicatas usando tabela temporária + MERGE"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    
    df = df.loc[:, ~df.columns.duplicated()]
    
    # 1. Criar tabela temporária
    tabela_temp = f"{tabela}_temp"
    df.to_sql(tabela_temp, con=engine, if_exists="replace", index=False, chunksize=500)
    print(f"📥 Dados temporários carregados ({len(df)} registros)")
    
    # 2. Fazer MERGE (inserir apenas novos registros)
    query_merge = f"""
    MERGE INTO {tabela} AS target
    USING {tabela_temp} AS source
    ON target.ID = source.ID
    WHEN NOT MATCHED THEN
        INSERT ({', '.join(df.columns)})
        VALUES ({', '.join(['source.' + col for col in df.columns])})
    WHEN MATCHED THEN
        UPDATE SET {', '.join([f'target.{col} = source.{col}' for col in df.columns if col != 'ID'])};
    """
    
    with engine.begin() as conn:
        result = conn.execute(text(query_merge))
        conn.execute(text(f"DROP TABLE {tabela_temp}"))
    
    print(f"💾 Dados sincronizados na tabela {tabela} (novos + atualizados)!")

# ================================
# EXECUÇÃO - MÊS ATUAL
# ================================
if __name__ == "__main__":
    # Pega o mês e ano atuais automaticamente
    hoje = datetime.now()
    ANO_ATUAL = hoje.year
    MES_ATUAL = hoje.month
    MAX_THREADS = 5

    print(f"🚀 Iniciando coleta do mês atual: {MES_ATUAL}/{ANO_ATUAL}")

    if not testar_token():
        print("❌ Corrija o token antes de continuar")
    else:
        criar_tabela_se_nao_existe()  # Garante que a tabela existe
        
        df_total = pd.DataFrame()
        for empresa in EMPRESAS:
            print(f"\n🏢 Coletando dados para empresa: {empresa}")
            df_mes = coletar_mes_atual_multithread(ANO_ATUAL, MES_ATUAL, empresa, max_workers=MAX_THREADS)
            
            if not df_mes.empty:
                df_mes['Empresa'] = empresa
                df_mes = df_mes.loc[:, ~df_mes.columns.duplicated()]
                df_total = pd.concat([df_total, df_mes], ignore_index=True)
            else:
                print(f"⚠️ Nenhum dado coletado para a empresa {empresa}")

        armazenar_no_sql_incremental(df_total, tabela="pedidos")
        print(f"\n🏁 Processo finalizado! Total de registros coletados: {len(df_total)}")

🚀 Iniciando coleta do mês atual: 3/2026
✅ Token válido!
✅ Tabela 'pedidos' verificada/criada

🏢 Coletando dados para empresa: CASA DE SUCO - BARRA DO CORDA
📅 Coletando dados de 31 dias do mês 3/2026 para empresa CASA DE SUCO - BARRA DO CORDA
📦 Total de registros no dia 2026-03-04 para empresa CASA DE SUCO - BARRA DO CORDA: 135
📦 Total de registros no dia 2026-03-01 para empresa CASA DE SUCO - BARRA DO CORDA: 234
📦 Total de registros no dia 2026-03-02 para empresa CASA DE SUCO - BARRA DO CORDA: 175
📦 Total de registros no dia 2026-03-05 para empresa CASA DE SUCO - BARRA DO CORDA: 164
📦 Total de registros no dia 2026-03-03 para empresa CASA DE SUCO - BARRA DO CORDA: 131
📦 Total de registros no dia 2026-03-07 para empresa CASA DE SUCO - BARRA DO CORDA: 195
📦 Total de registros no dia 2026-03-08 para empresa CASA DE SUCO - BARRA DO CORDA: 152📦 Total de registros no dia 2026-03-06 para empresa CASA DE SUCO - BARRA DO CORDA: 177

📦 Total de registros no dia 2026-03-10 para empresa CASA DE SU